# ResNet152V2 — complete training notebook

This notebook contains the complete data preparation, model construction, training, evaluation and export workflow. No training module outside this notebook is used. Run cells in order only after reviewing them.

**Goal:** train the standard ResNet152V2 architecture locally from random initialization and measure whether extreme depth helps this dataset. The final cell registers the artifact so the backend and model selector can use it.


## End-to-end flow

```text
image files → parallel decode/resize → augmentation → model → pneumonia probability
            → untouched test metrics → .keras artifact → web application
```

The test set is isolated until final evaluation. This prevents test information leaking into model selection.


## 1. Paths and experiment controls

The seed fixes the data split and initial weights. Image size, batch size, epochs and learning rate are explicit so runtime and model quality can be tuned from one cell.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import random

import numpy as np
import tensorflow as tf

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASET = ROOT / "dataset" / "archive" / "chest_xray"
CATALOG = ROOT / "server" / "model_catalog.json"
ARTIFACT = ROOT / "server" / "artifacts" / "resnet152v2.keras"
MODEL_ID = "resnet152v2"
IMAGE_SIZE = 224
BATCH_SIZE = 12
EPOCHS = 35
LEARNING_RATE = 0.0001
SEED = 42

assert DATASET.is_dir(), f"Dataset not found: {DATASET}"
ARTIFACT.parent.mkdir(parents=True, exist_ok=True)


## 2. CPU/GPU execution and multithreading

TensorFlow parallelizes numerical kernels. The explicit thread pools use available CPU cores, while `AUTOTUNE` below overlaps file decoding and preprocessing with model execution. Training multiple models simultaneously would normally cause RAM/GPU contention, so each notebook trains one model efficiently.


In [2]:
# Configure TensorFlow before it creates its execution context.
CPU_THREADS = max(1, (os.cpu_count() or 2) - 1)
try:
    tf.config.threading.set_intra_op_parallelism_threads(CPU_THREADS)
    tf.config.threading.set_inter_op_parallelism_threads(max(1, CPU_THREADS // 2))
except RuntimeError:
    print("Thread pools already initialized; restart the kernel to change them.")

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print({"cpu_threads": CPU_THREADS, "gpus": tf.config.list_physical_devices("GPU")})


{'cpu_threads': 19, 'gpus': []}


## 3. Discover and split the data

The training and small validation folders are combined, then split 80/20 within each class. Stratification preserves the Normal/Pneumonia ratio. Seed 42 makes the assignment repeatable. The original test folder remains untouched.


In [3]:
EXTENSIONS = {".jpg", ".jpeg", ".png"}

def image_files(split, class_name, label):
    folder = DATASET / split / class_name
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {folder}")
    return [(str(path), label) for path in sorted(folder.iterdir()) if path.suffix.lower() in EXTENSIONS]

pool = image_files("train", "NORMAL", 0) + image_files("train", "PNEUMONIA", 1)
pool += image_files("val", "NORMAL", 0) + image_files("val", "PNEUMONIA", 1)
test_samples = image_files("test", "NORMAL", 0) + image_files("test", "PNEUMONIA", 1)

rng = random.Random(SEED)
train_samples, validation_samples = [], []
for label in (0, 1):
    group = [sample for sample in pool if sample[1] == label]
    rng.shuffle(group)
    cut = round(len(group) * 0.2)
    validation_samples.extend(group[:cut])
    train_samples.extend(group[cut:])
rng.shuffle(train_samples)
rng.shuffle(validation_samples)

def counts(samples):
    return {"normal": sum(label == 0 for _, label in samples), "pneumonia": sum(label == 1 for _, label in samples)}

{"train": counts(train_samples), "validation": counts(validation_samples), "test": counts(test_samples)}


{'train': {'normal': 1079, 'pneumonia': 3106},
 'validation': {'normal': 270, 'pneumonia': 777},
 'test': {'normal': 234, 'pneumonia': 390}}

## 4. Build the parallel input pipeline

Decoding and resizing run on multiple CPU threads. Prefetch prepares the next batch while the model processes the current one. Balanced class weights stop majority-class accuracy from hiding poor Normal recognition.


In [4]:
def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    image = tf.image.resize(tf.cast(image, tf.float32), (IMAGE_SIZE, IMAGE_SIZE)) / 255.0
    return image, tf.cast(label, tf.float32)

def make_dataset(samples, training=False):
    paths, labels = zip(*samples)
    data = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    if training:
        data = data.shuffle(len(samples), seed=SEED, reshuffle_each_iteration=True)
    return (data
            .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

train_data = make_dataset(train_samples, training=True)
validation_data = make_dataset(validation_samples)
test_data = make_dataset(test_samples)

class_counts = np.bincount([label for _, label in train_samples], minlength=2)
class_weight = {index: len(train_samples) / (2 * count) for index, count in enumerate(class_counts)}
class_weight


{0: np.float64(1.9392956441149212), 1: np.float64(0.6736960721184804)}

## 5. Define the model from random initialization

ResNet's identity shortcuts let gradients bypass many nonlinear layers, which makes a 152-layer network trainable. The backbone uses no downloaded weights. Global average pooling sharply reduces the classifier size; the smaller batch fits the larger model in memory.

```text
224² RGB → augmentation → ResNet152V2 (no external weights) → GlobalAvgPool → dropout → sigmoid
```


In [5]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.035),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomContrast(0.08),
], name="augmentation")

backbone = tf.keras.applications.ResNet152V2(
    include_top=False,
    weights=None,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
)
inputs = tf.keras.layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
x = augmentation(inputs)
x = backbone(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.35)(x)
model = tf.keras.Model(inputs, tf.keras.layers.Dense(1, activation="sigmoid")(x), name=MODEL_ID)


## 6. Compile

Binary cross-entropy trains a calibrated two-class probability. Adam adapts parameter updates. Accuracy is intuitive; AUC measures ranking quality across thresholds.


In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.AUC(name="auc")],
)
model.summary()
print({"parameters": model.count_params(), "artifact": str(ARTIFACT)})


Model: "resnet152v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 7, 7, 2048)     │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,333,697 (222.53 MB)

 Trainable params: 58,189,953 (221.98 MB)

 Non-trainable params: 143,744 (561.50 KB)

{'parameters': 58333697, 'artifact': 'C:\\Users\\praloya\\Desktop\\New folder\\projects\\Breathe-AI\\server\\artifacts\\resnet152v2.keras'}


## 7. Train — long-running cell

Checkpointing retains the best validation-loss model. Early stopping avoids wasted epochs after progress stalls. Learning-rate reduction makes smaller, more precise updates near a good solution. Augmentation is enabled only for training.


In [7]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ARTIFACT, monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.3, min_lr=1e-7),
]

history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)
print({
    "epochs_completed": len(history.history["loss"]),
    "best_validation_loss": min(history.history["val_loss"]),
    "best_validation_accuracy": max(history.history["val_accuracy"]),
})


Epoch 1/35


c:\Users\praloya\Desktop\New folder\projects\Breathe-AI\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


349/349 ━━━━━━━━━━━━━━━━━━━━ 2002s 6s/step - accuracy: 0.8528 - auc: 0.9258 - loss: 0.3470 - val_accuracy: 0.7421 - val_auc: 0.5000 - val_loss: 2.9578 - learning_rate: 1.0000e-04
Epoch 2/35
349/349 ━━━━━━━━━━━━━━━━━━━━ 1786s 5s/step - accuracy: 0.8939 - auc: 0.9596 - loss: 0.2557 - val_accuracy: 0.5788 - val_auc: 0.8348 - val_loss: 1.3734 - learning_rate: 1.0000e-04
Epoch 3/35
349/349 ━━━━━━━━━━━━━━━━━━━━ 1928s 6s/step - accuracy: 0.9176 - auc: 0.9746 - loss: 0.2027 - val_accuracy: 0.2607 - val_auc: 0.8279 - val_loss: 2.3016 - learning_rate: 1.0000e-04
Epoch 4/35
349/349 ━━━━━━━━━━━━━━━━━━━━ 1753s 5s/step - accuracy: 0.9324 - auc: 0.9807 - loss: 0.1730 - val_accuracy: 0.2808 - val_auc: 0.8845 - val_loss: 2.4677 - learning_rate: 1.0000e-04
Epoch 5/35
349/349 ━━━━━━━━━━━━━━━━━━━━ 1642s 5s/step - accuracy: 0.9395 - auc: 0.9826 - loss: 0.1622 - val_accuracy: 0.6724 - val_auc: 0.9589 - val_loss: 0.8267 - learning_rate: 1.0000e-04
Epoch 6/35
349/349 ━━━━━━━━━━━━━━━━━━━━ 1441s 4s/step - accur

## 8. Evaluate on untouched images

The comparison includes accuracy, loss, precision, recall/sensitivity, specificity, F1, AUC and a confusion matrix. Threshold 0.5 is stored with the metrics so every model is judged consistently.


In [8]:
model = tf.keras.models.load_model(ARTIFACT, compile=False)
probabilities = np.asarray(model.predict(test_data, verbose=1)).reshape(-1)
actual = np.concatenate([labels.numpy() for _, labels in test_data]).astype(int)
predicted = (probabilities >= 0.5).astype(int)

tn = int(np.sum((actual == 0) & (predicted == 0)))
fp = int(np.sum((actual == 0) & (predicted == 1)))
fn = int(np.sum((actual == 1) & (predicted == 0)))
tp = int(np.sum((actual == 1) & (predicted == 1)))
divide = lambda numerator, denominator: float(numerator / denominator) if denominator else 0.0
precision = divide(tp, tp + fp)
recall = divide(tp, tp + fn)
clipped = np.clip(probabilities, 1e-7, 1 - 1e-7)

metrics = {
    "status": "verified_local_test_split",
    "accuracy": divide(tp + tn, len(actual)),
    "loss": float(-np.mean(actual * np.log(clipped) + (1 - actual) * np.log(1 - clipped))),
    "precision": precision,
    "recall": recall,
    "f1": divide(2 * precision * recall, precision + recall),
    "specificity": divide(tn, tn + fp),
    "auc": float(tf.keras.metrics.AUC()(actual, probabilities).numpy()),
    "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
    "samples": len(actual),
    "threshold": 0.5,
}
metrics


52/52 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step


{'status': 'verified_local_test_split',
 'accuracy': 0.7884615384615384,
 'loss': 0.8411999007353862,
 'precision': 0.749034749034749,
 'recall': 0.9948717948717949,
 'f1': 0.854625550660793,
 'specificity': 0.4444444444444444,
 'auc': 0.9089469313621521,
 'confusion_matrix': {'tn': 104, 'fp': 130, 'fn': 2, 'tp': 388},
 'samples': 624,
 'threshold': 0.5}

## 9. Save and register for inference

This cell hashes the best artifact and writes its verified metrics and parameter count to the catalog. The backend verifies the hash before loading it.


In [9]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

digest = sha256(ARTIFACT)
catalog = json.loads(CATALOG.read_text(encoding="utf-8"))
entry = next(item for item in catalog["models"] if item["id"] == MODEL_ID)
entry["parameters"] = int(model.count_params())
entry["metrics"] = metrics
entry["artifact"].update({
    "version": digest[:12],
    "sha256": digest,
    "status": "trained_and_checksum_verified",
    "trained_at": datetime.now(timezone.utc).isoformat(),
})
CATALOG.write_text(json.dumps(catalog, indent=2) + "\n", encoding="utf-8")
print({"saved": str(ARTIFACT), "model_version": digest[:12], "catalog_updated": str(CATALOG)})


{'saved': 'C:\\Users\\praloya\\Desktop\\New folder\\projects\\Breathe-AI\\server\\artifacts\\resnet152v2.keras', 'model_version': '80b34f071f72', 'catalog_updated': 'C:\\Users\\praloya\\Desktop\\New folder\\projects\\Breathe-AI\\server\\model_catalog.json'}


## What to do after running

Restart the backend, refresh the web application, choose **ResNet152V2**, and upload an image. Retraining this notebook replaces only this model's artifact and catalog metrics.
